# Kaggle worker 3 — FUnIE-GAN, U-Net, and Water-Net
Runs the three lighter reference methods on UIEB and LSUI for selected seeds, retaining their separate method-specific training formulations.

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/heniath/underwater-image-enhancement.git'; REPO_BRANCH='feature/epoch-logging'
REPO_DIR=Path('/kaggle/working/underwater-image-enhancement')
UIEB_INPUT=Path('/kaggle/input/datasets/ohmahler91/uieb-dataset'); LSUI_INPUT=Path('/kaggle/input/datasets/ohmahler91/lsui-dataset/LSUI')
DATA_ROOT=Path('/kaggle/working/reference_data'); OUTPUT_ROOT=Path('/kaggle/working/reference_outputs_person3_classic'); TORCH_CACHE=Path('/kaggle/working/torch_cache')
SMOKE_BASELINE_INPUT=None  # Set to an attached completed smoke output when available.
METHODS=['funie_gan','unet','water_net']; SEEDS=[0,1,2]  # Split methods/seeds further across accounts if desired.
RUN_TESTS=True; RUN_WORKER=False; USE_RAM_CACHE=True

## Setup repository, environment, and datasets

In [ ]:
import csv, importlib, os, shutil, subprocess, sys
if not (REPO_DIR/'.git').exists(): subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',REPO_BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','-B',REPO_BRANCH,f'origin/{REPO_BRANCH}'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only','origin',REPO_BRANCH],check=True)
subprocess.run([sys.executable,'-u','-m','pip','install','-q','-e',f'{REPO_DIR}[dev,profile,visualization]'],check=True); os.chdir(REPO_DIR)
repo_src=str(REPO_DIR/'src'); sys.path.insert(0,repo_src) if repo_src not in sys.path else None; importlib.invalidate_caches()
import torch, uwir
assert torch.cuda.is_available(),'Enable a Kaggle GPU accelerator.'
print('GPU:',torch.cuda.get_device_name(0)); print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()); print('uwir:',Path(uwir.__file__).resolve())
TORCH_CACHE.mkdir(parents=True,exist_ok=True); os.environ['UWIR_TORCH_HOME']=str(TORCH_CACHE)
def find_uieb(root):
    for c in [root]+[p.parent for p in root.rglob('raw-890')]:
        if (c/'raw-890').is_dir() and (c/'reference-890').is_dir(): return c.resolve()
    raise FileNotFoundError(f'UIEB layout not found below {root}')
def link(name,target):
    p=DATA_ROOT/name
    if p.is_symlink(): p.unlink()
    elif p.exists(): raise FileExistsError(f'Refusing to replace {p}')
    p.symlink_to(target,target_is_directory=True)
if not UIEB_INPUT.is_dir() or not LSUI_INPUT.is_dir(): raise FileNotFoundError('Attach both datasets and correct their paths.')
DATA_ROOT.mkdir(parents=True,exist_ok=True); link('UIEB',find_uieb(UIEB_INPUT)); link('LSUI',LSUI_INPUT.resolve())
if SMOKE_BASELINE_INPUT is not None and Path(SMOKE_BASELINE_INPUT).is_dir() and not OUTPUT_ROOT.exists(): shutil.copytree(SMOKE_BASELINE_INPUT,OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True); print('UIEB:',(DATA_ROOT/'UIEB').resolve()); print('LSUI:',(DATA_ROOT/'LSUI').resolve())

## Strict discovery and tests

In [ ]:
from uwir.datasets.lsui import discover_lsui
from uwir.datasets.uieb import discover_uieb
uieb=discover_uieb(DATA_ROOT/'UIEB'); lsui,report=discover_lsui(DATA_ROOT/'LSUI')
print('UIEB pairs:',len(uieb)); print('LSUI:',report)
if RUN_TESTS: subprocess.run([sys.executable,'-u','-m','pytest','-q'],cwd=REPO_DIR,check=True)

## Smoke gate and assigned full runs
Set `RUN_WORKER=True`. For more accounts, divide `METHODS` and/or `SEEDS` into non-overlapping shards. Keep each account's output separate.

In [ ]:
from uwir.reference_methods import REFERENCE_METHODS
def command(mode,methods=None,seeds=None):
    cmd=[sys.executable,'-u','-m','scripts.reference_methods_benchmark',f'--{mode}','--device','cuda','--data-root',str(DATA_ROOT),'--output-root',str(OUTPUT_ROOT)]
    if not USE_RAM_CACHE: cmd.append('--no-ram-cache')
    if methods: cmd += ['--methods',*methods]
    if seeds is not None: cmd += ['--seeds',*map(str,seeds)]
    print(' '.join(cmd)); subprocess.run(cmd,cwd=REPO_DIR,env=os.environ.copy(),check=True)
def smoke_complete():
    p=OUTPUT_ROOT/'smoke_results.csv'
    if not p.exists(): return False
    with p.open(newline='',encoding='utf-8') as h: found={(r['dataset'],r['method']) for r in csv.DictReader(h)}
    return found=={(d,m) for d in ('UIEB','LSUI') for m in REFERENCE_METHODS}
if RUN_WORKER:
    if not smoke_complete(): command('smoke')
    if not smoke_complete(): raise RuntimeError('The complete smoke matrix did not pass.')
    command('full',METHODS,SEEDS)
else: print('Ready. Set RUN_WORKER=True to launch:',METHODS,SEEDS)

## Progress and handoff

In [ ]:
import pandas as pd
p=OUTPUT_ROOT/'per_run_results.csv'
if p.exists(): display(pd.read_csv(p).sort_values(['dataset','method','seed']))
print('Completed assigned runs:',len(list(OUTPUT_ROOT.glob('*/*/seed_*/test_metrics.json'))),'/',len(METHODS)*2*len(SEEDS)); print('Save as a private Kaggle Dataset:',OUTPUT_ROOT)